# Convolutional Hybrid QNN - Binarized MNIST

This notebook trains a small hybrid CNN+QNN on the binarized MNIST dataset (threshold 0.5).

In [5]:
import os
import random
import copy
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s):
    os.environ['PYTHONHASHSEED'] = str(s)
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [6]:
BASE_DIR = Path('/home/sammarv/quantum_corrosion')
OUT_DIR = BASE_DIR / 'results/binarized_mnist_conv_hybrid_qnn'
OUT_DIR.mkdir(parents=True, exist_ok=True)
# Try qp dataset first, fallback to torchvision MNIST binarized
try:
    import qp
    data = qp.data.load('other', name='binarized-mnist')
    X_all = np.array(data['X']).astype(np.float32)
    y_all = np.array(data['y']).astype(np.int64)
    if X_all.ndim == 4 and X_all.shape[1] == 1:
        X_all = X_all.squeeze(1)
    print('Loaded via qp:', X_all.shape)
except Exception as e:
    # qp or torchvision import failed (binary mismatch). Fallback to sklearn's openml loader
    print('qp failed or torchvision unavailable:', repr(e))
    from sklearn.datasets import fetch_openml
    mnist = fetch_openml('mnist_784', version=1, as_frame=False)
    X = np.array(mnist['data'], dtype=np.float32).reshape(-1, 28, 28) / 255.0
    y = np.array(mnist['target'], dtype=np.int64)
    # Binarize (threshold 0.5)
    X_all = (X > 0.5).astype(np.float32)
    y_all = y
N_CLASSES = len(np.unique(y_all))
print('N_CLASSES', N_CLASSES)

qp failed or torchvision unavailable: ModuleNotFoundError("No module named 'qp'")
N_CLASSES 10


In [7]:
# Minimal prepare (no augmentation)
X_all_prep = X_all.astype(np.float32)
y_all_prep = y_all.astype(np.int64)
print('Prepared shapes:', X_all_prep.shape, y_all_prep.shape)

Prepared shapes: (70000, 28, 28) (70000,)


In [8]:
# Preprocess: flatten -> standardize -> reshape for Conv2d
N = len(X_all_prep); H, W = 28, 28
X_flat = X_all_prep.reshape(N, -1)
scaler = StandardScaler(); X_scaled = scaler.fit_transform(X_flat).reshape(N, H, W).astype(np.float32)
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(X_scaled, y_all_prep, test_size=0.2, random_state=SEED, stratify=y_all_prep)
X_train_t = torch.tensor(X_train_np[:, None, :, :], dtype=torch.float32)
y_train_t = torch.tensor(y_train_np, dtype=torch.long)
X_test_t = torch.tensor(X_test_np[:, None, :, :], dtype=torch.float32)
y_test_t = torch.tensor(y_test_np, dtype=torch.long)
batch_size = 128
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=batch_size, shuffle=False)
print('Train/Test:', X_train_t.shape, X_test_t.shape)

Train/Test: torch.Size([56000, 1, 28, 28]) torch.Size([14000, 1, 28, 28])


In [9]:
# Quantum node + small batched wrapper
n_qubits = 4; n_layers = 2
def _make_quantum_device(n_q):
    try:
        dev = qml.device('lightning.gpu', wires=n_q)
        return dev, 'adjoint'
    except Exception:
        dev = qml.device('default.qubit', wires=n_q)
        return dev, 'backprop'
dev, diff_method = _make_quantum_device(n_qubits)
@qml.qnode(dev, interface='torch', diff_method=diff_method)
def qnode(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
class BatchedQuantumLayer(nn.Module):
    def __init__(self, n_l, n_q, q_node):
        super().__init__()
        self.n_qubits = n_q
        self.qnode = q_node
        self.weights = nn.Parameter(torch.empty(n_l, n_q).uniform_(-np.pi, np.pi))
    def forward(self, x):
        # x: (B, n_qubits)
        out = self.qnode(x, self.weights)
        # qnode may return a Python list of length n_qubits (each element length B),
        # or a tensor. Normalize to a torch.Tensor of shape (n_qubits, B).
        if isinstance(out, list):
            out = torch.stack([torch.tensor(o) if not isinstance(o, torch.Tensor) else o for o in out])
        if not isinstance(out, torch.Tensor):
            out = torch.tensor(out)
        out = out.float().view(self.n_qubits, x.shape[0]).t().contiguous()
        return out

In [10]:
# Model: small CNN -> classical skip -> quantum projection -> fusion
class QuantumGuidedFusion(nn.Module):
    def __init__(self, c_dim=64, q_dim=4, n_classes=10):
        super().__init__()
        self.film_net = nn.Sequential(nn.Linear(q_dim,32), nn.GELU(), nn.Linear(32, c_dim*2))
        self.gate = nn.Sequential(nn.Linear(c_dim+q_dim, c_dim), nn.Sigmoid())
        self.classifier = nn.Linear(c_dim, n_classes)
    def forward(self, c, q):
        film_params = self.film_net(q); gamma, beta = torch.chunk(film_params, 2, dim=1)
        c_mod = c * (gamma + 1.0) + beta
        gate_val = self.gate(torch.cat([c_mod, q], dim=1))
        fused = gate_val * c_mod + (1-gate_val) * c
        return self.classifier(fused)
class ConvolutionalHybridQNN(nn.Module):
    def __init__(self, n_cls=None):
        super().__init__()
        n_classes = n_cls if n_cls is not None else 10
        self.cnn = nn.Sequential(nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
                               nn.Flatten(), nn.Linear(32*7*7,256), nn.BatchNorm1d(256), nn.ReLU())
        self.qnn_proj = nn.Sequential(nn.Linear(256, n_qubits), nn.Sigmoid())
        self.qnn = BatchedQuantumLayer(n_layers, n_qubits, qnode)
        self.classical_skip = nn.Sequential(nn.Linear(256,64), nn.ReLU())
        self.fusion = QuantumGuidedFusion(c_dim=64, q_dim=n_qubits, n_classes=n_classes)
    def forward(self, x):
        feats = self.cnn(x)
        q_in = self.qnn_proj(feats) * (2.0 * np.pi)
        q_feats = self.qnn(q_in)
        c_feats = self.classical_skip(feats)
        return self.fusion(c_feats, q_feats)
model = ConvolutionalHybridQNN(n_cls=None).to(device)
print('Model params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

Model params: 434006


In [11]:
# Training loop (short run: epochs=1 for quick execution)
epochs = 1
class_counts = np.bincount(y_train_np)
cw = 1.0 / class_counts.astype(np.float32); cw = cw / cw.sum() * len(class_counts)
loss_weights = torch.tensor(cw, dtype=torch.float32).to(device)
opt = optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss(weight=loss_weights)
best_f1 = 0.0; best_weights = None
for epoch in range(epochs):
    model.train()
    running = 0.0
    for Xb, yb in train_loader:
        Xb = Xb.to(device); yb = yb.to(device)
        opt.zero_grad(); out = model(Xb); loss = crit(out, yb); loss.backward(); opt.step()
        running += loss.item() * Xb.size(0)
    train_loss = running / len(train_loader.dataset)
    model.eval(); preds=[]; targs=[]
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device); yb = yb.to(device)
            out = model(Xb); preds.extend(torch.argmax(out, dim=1).cpu().numpy()); targs.extend(yb.cpu().numpy())
    acc = accuracy_score(targs, preds); f1 = f1_score(targs, preds, average='macro', zero_division=0)
    print(f'Epoch {epoch+1}/{epochs} - train_loss={train_loss:.4f} val_acc={acc:.4f} val_f1={f1:.4f}')
    if f1 > best_f1: best_f1 = f1; best_weights = copy.deepcopy(model.state_dict())

Epoch 1/1 - train_loss=0.1734 val_acc=0.9817 val_f1=0.9816


In [12]:
# Evaluation and save
if best_weights is not None: model.load_state_dict(best_weights)
model.eval(); preds=[]; targs=[]
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        out = model(Xb)
        preds.extend(torch.argmax(out, dim=1).cpu().numpy()); targs.extend(yb.numpy())
print('Final acc:', accuracy_score(targs, preds))
print(classification_report(targs, preds))
torch.save(model.state_dict(), str(OUT_DIR / 'conv_hybrid_qnn_best.pt'))
print('Saved:', OUT_DIR / 'conv_hybrid_qnn_best.pt')

Final acc: 0.9817142857142858
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1381
           1       0.98      0.99      0.99      1575
           2       0.99      0.97      0.98      1398
           3       1.00      0.97      0.98      1428
           4       0.98      0.98      0.98      1365
           5       0.97      0.98      0.97      1263
           6       0.98      0.99      0.99      1375
           7       0.97      0.99      0.98      1459
           8       0.98      0.97      0.98      1365
           9       0.97      0.98      0.98      1391

    accuracy                           0.98     14000
   macro avg       0.98      0.98      0.98     14000
weighted avg       0.98      0.98      0.98     14000

Saved: /home/sammarv/quantum_corrosion/results/binarized_mnist_conv_hybrid_qnn/conv_hybrid_qnn_best.pt
